In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "retail_analytics"

orders = spark.table(f"{CATALOG}.{SCHEMA}.bronze_orders")
products = spark.table(f"{CATALOG}.{SCHEMA}.bronze_products")

print("Orders columns:", orders.columns)
print("Products columns:", products.columns)

In [ ]:
# Normalizar tipos y limpiar órdenes.
# Ajusta los nombres de columnas al esquema real de los datasets.
silver_orders = (
    orders
    .withColumn("order_id", F.col("order_id").cast("string"))
    .withColumn("product_id", F.col("product_id").cast("string"))
    .withColumn("order_date", F.to_date(F.col("order_date")))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .dropDuplicates(["order_id", "product_id"])
    .filter(
        F.col("order_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("order_date").isNotNull()
        & (F.col("quantity") > 0)
        & (F.col("unit_price") >= 0)
    )
)

silver_products = (
    products
    .withColumn("product_id", F.col("product_id").cast("string"))
    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.trim(F.col("category")))
    .dropDuplicates(["product_id"])
    .filter(
        F.col("product_id").isNotNull()
        & F.col("product_name").isNotNull()
    )
)

display(silver_orders.limit(10))
display(silver_products.limit(10))

In [ ]:
(
    silver_orders.write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_orders")
)

(
    silver_products.write.format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_products")
)

print("Silver orders:", spark.table(f"{CATALOG}.{SCHEMA}.silver_orders").count())
print("Silver products:", spark.table(f"{CATALOG}.{SCHEMA}.silver_products").count())